# Почему изменилась выручка ПРАЙМ

Сравниваем два соседних окна по семь завершённых дней UTC на данных твоего сегмента. Модель из С6 умножает число успешных платежей на средний платёж. Сначала меняем число платежей, затем средний платёж.

In [ ]:
from pathlib import Path
import sys
from datetime import date, timedelta

import pandas as pd

notebooks_dir = Path.cwd() / "notebooks"
if notebooks_dir.is_dir() and str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

from prime_analytics_data import load_daily

import matplotlib.pyplot as plt

from prime.factors import weekly_revenue_factors

report_date = date(2026, 6, 10)
week = timedelta(days=7)
daily, _ = load_daily(report_date - 2 * week, report_date)
display(daily[["payments_count", "total_revenue", "payments_complete"]])

In [ ]:
factors = weekly_revenue_factors(daily, report_date)
factor_keys = [
    "previous_revenue", "payment_count_effect", "average_payment_effect", "current_revenue",
]
if factors["issues"]:
    print("Причины пропуска разложения:", factors["issues"])
else:
    display(pd.Series({name: factors[name] for name in factor_keys}, name="₽"))

In [ ]:
if not factors["issues"]:
    r0, count_effect, average_effect, r1 = (factors[name] for name in factor_keys)
    waterfall = pd.DataFrame({
        "height": [r0, count_effect, average_effect, r1],
        "bottom": [0, r0, r0 + count_effect, 0],
    }, index=["Прошлые 7 дней", "Число платежей", "Средний платёж", "Последние 7 дней"])

    fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
    colors = ["#58708f", *["#238875" if x >= 0 else "#b74b4b" for x in (count_effect, average_effect)], "#58708f"]
    bars = ax.bar(waterfall.index, waterfall["height"], bottom=waterfall["bottom"], color=colors)
    labels = [f"{r0:,.0f}", f"{count_effect:+,.0f}", f"{average_effect:+,.0f}", f"{r1:,.0f}"]
    ax.bar_label(bars, labels=[label.replace(",", " ") for label in labels], padding=3)
    ax.set_ylim(bottom=0)
    ax.margins(y=0.15)
    ax.set_ylabel("₽")
    ax.set_title("Изменение выручки за две соседние недели")
    plt.show()

Для отчёта передаём границы окон и четыре суммы из `factors`. При неполной выгрузке сообщаем, какое окно нельзя сравнить. Вклад среднего платежа может зависеть от тарифа и структуры покупок.